# D · Trajectory Forecasting with Qwen3.6-27B (team Qwen setting)

1. Add the Qwen secrets (key icon on the left; switch **Notebook access** on):
   `QWEN_BASE_URL` (like `https://<resource>.services.ai.azure.com/openai/v1/`), `QWEN_AZURE_API_KEY`,
   and optionally `QWEN_DEPLOYMENT` (default `qwen--qwen36-27b`).
2. Click **Runtime → Run all**. Keep this tab open.

What it runs: 3 dev cases, then the same 100 test_core cases as GPT-5-mini, input = state_with_history with the frozen
GPT-derived state (A's file). Setting = team rule: max_tokens 8192 (one retry at 16384 if cut off), no reasoning_effort /
temperature / top_p / response_format, timeout 600 s, 0 automatic retries.
Each case takes about 2–4 minutes, so the 100 cases take about 1.5–3 hours. Every finished case is saved; if Colab
disconnects, just **Run all** again and it continues where it stopped.

In [ ]:
# Step 1: find the files on Drive, unzip the data, install the tools, copy in the Qwen code update
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil, subprocess, sys
roots = glob.glob('/content/drive/MyDrive/**/D_trajectory/code/run_trajectory_llm.py', recursive=True)
zips = glob.glob('/content/drive/MyDrive/**/data_v1_share_v1.2.zip', recursive=True)
assert roots, 'Cannot find the D_trajectory folder in My Drive.'
assert zips, 'Cannot find data_v1_share_v1.2.zip in My Drive.'
ROOT = os.path.dirname(os.path.dirname(roots[0]))
for f in glob.glob(os.path.join(ROOT, 'code_qwen_update', '*.py')):   # newer code with the Qwen mode
    shutil.copy(f, os.path.join(ROOT, 'code', os.path.basename(f)))
assert 'provider' in open(os.path.join(ROOT, 'code', 'run_trajectory_llm.py')).read(), 'Qwen code update missing (folder code_qwen_update).'
for need in ['inputs/state_predictions_A.jsonl', 'inputs/state_predictions_dev_C.jsonl']:
    assert os.path.exists(os.path.join(ROOT, need)), f'Missing {need} in D_trajectory'
DATA = '/content/data_v1_share'
if not os.path.exists(DATA + '/labels.csv'):
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content/'], check=True)
print('ROOT =', ROOT); print('DATA =', DATA)
!pip -q install "openai==2.54.0" "scikit-survival==0.28.0"
%cd {ROOT}/code
print('Step 1 OK')

In [ ]:
# Step 2: read the Qwen secrets (never typed into the notebook)
from google.colab import userdata
def get(k):
    try:
        return (userdata.get(k) or '').strip()
    except Exception:
        return ''
for k in ['QWEN_BASE_URL', 'QWEN_AZURE_API_KEY']:
    assert get(k), f'Secret {k} is missing. Add it with the key icon on the left and switch Notebook access on.'
    os.environ[k] = get(k)
os.environ['QWEN_DEPLOYMENT'] = get('QWEN_DEPLOYMENT') or 'qwen--qwen36-27b'
print('Step 2 OK: deployment =', os.environ['QWEN_DEPLOYMENT'])

In [ ]:
# Step 3: one tiny test call (checks endpoint, key, and that the model really is Qwen3.6-27B)
def run(args):
    p = subprocess.Popen([sys.executable, '-u', 'run_trajectory_llm.py', '--provider', 'qwen', '--root', ROOT, '--data', DATA] + args,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end=''); out.append(line)
    p.wait()
    return p.returncode, ''.join(out)
rc, out = run(['--ping'])
assert rc == 0 and 'connection ok' in out, 'Qwen test failed. Copy the message above and send it to Claude.'
print('Step 3 OK')

In [ ]:
# Step 4: 3 dev cases (check only, not scored), then the 100 test_core cases (state + history, then history only)
STATE_DEV = os.path.join(ROOT, 'inputs', 'state_predictions_dev_C.jsonl')
STATE = os.path.join(ROOT, 'inputs', 'state_predictions_A.jsonl')
rc, out = run(['--split', 'dev', '--workers', '1', '--state', STATE_DEV, '--state-version', 'C_dev_gpt5mini'])
assert rc == 0 and 'ok 0/' not in out, 'Dev run failed. Send the message above to Claude.'
rc, out = run(['--split', 'core', '--workers', '2', '--state', STATE, '--state-version', 'A_v1'])
assert rc == 0, 'Core run stopped. Send the message above to Claude.'
rc, out = run(['--split', 'core', '--workers', '2'])   # history only, same as the GPT runs
assert rc == 0, 'History-only run stopped. Send the message above to Claude.'
print('Step 4 OK')

In [ ]:
# Step 4b (optional): if some cases failed only because of a timeout / connection error ("failed: ... api: ..."),
# change False to True and run this cell once. This is an explicit resume (as B did); failures stay in the log.
RETRY_FAILED = False
if RETRY_FAILED:
    rc, out = run(['--split', 'core', '--workers', '1', '--state', STATE, '--state-version', 'A_v1', '--retry-failed'])

In [ ]:
# Step 5: score, compare with GPT-5-mini on the same cases, plot
for args in [['score_trajectory.py', '--root', ROOT],
             ['compare_trajectory_models.py', '--root', ROOT,
              '--a', 'outputs/D/trajectory_predictions_qwen3.6-27b_state.jsonl',
              '--b', 'outputs/D/trajectory_predictions_gpt-5-mini_state.jsonl'],
             ['compare_trajectory_models.py', '--root', ROOT,
              '--a', 'outputs/D/trajectory_predictions_qwen3.6-27b_state.jsonl',
              '--b', 'outputs/D/trajectory_predictions_qwen3.6-27b.jsonl'],
             ['compare_trajectory_models.py', '--root', ROOT,
              '--a', 'outputs/D/trajectory_predictions_qwen3.6-27b.jsonl',
              '--b', 'outputs/D/trajectory_predictions_gpt-5-mini.jsonl'],
             ['plot_trajectory.py', '--root', ROOT]]:
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-1500:])
import pandas as pd
m = pd.read_csv(f'{ROOT}/metrics/D/trajectory_metrics.csv')
display(m[['method', 'model', 'input_setting', 'scored_n', 'failed_n', 'c_index', 'ibs']].round(3))
from IPython.display import Image, display as show
show(Image(f'{ROOT}/results/trajectory_survival_plot.png'))
print('ALL DONE. Take a screenshot of this page and send it to Claude.')